# DatetimeIndex Operations

The `DatetimeIndex` is the backbone of pandas time series — it enables date-based
slicing, resampling, and frequency-aware operations. Without a proper `DatetimeIndex`,
most of the powerful time series functionality in pandas (and in downstream libraries
like statsmodels) simply will not work.

This notebook provides a deep dive into creating, inspecting, and manipulating
`DatetimeIndex` objects. We will use the **airline passengers** dataset as a running
example throughout.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

DATA_DIR = Path("../data")

print(f"pandas {pd.__version__}")
print(f"numpy  {np.__version__}")

## 1. Creating a DatetimeIndex

There are three common ways to create a `DatetimeIndex`:

1. **`pd.date_range()`** — generate a sequence of dates programmatically.
2. **`pd.to_datetime()`** — convert an existing column or list of strings.
3. **`read_csv()` with `parse_dates` + `index_col`** — create it at load time.

### 1a. `pd.date_range()`

Specify any three of `start`, `end`, `periods`, and `freq`.  
Modern pandas 2.x aliases use `'ME'` (month-end), `'MS'` (month-start),
`'QE'` (quarter-end), `'YE'` (year-end), `'h'` (hour), `'min'` (minute).

In [ ]:
# Daily range
daily = pd.date_range(start="2024-01-01", periods=10, freq="D")
print("Daily:", daily)

In [ ]:
# Month-end range
monthly = pd.date_range(start="2024-01-01", end="2024-12-31", freq="ME")
print("Month-end:", monthly)

In [ ]:
# Quarter-end, year-end, hourly, and minute frequencies
print("Quarter-end:", pd.date_range("2024-01-01", periods=4, freq="QE"))
print("Year-end:   ", pd.date_range("2020-01-01", periods=5, freq="YE"))
print("Hourly:     ", pd.date_range("2024-03-01", periods=6, freq="h"))
print("Minutely:   ", pd.date_range("2024-03-01", periods=6, freq="min"))

### 1b. `pd.to_datetime()` applied to a column

In [ ]:
# Simulate a DataFrame with a string date column
raw = pd.DataFrame({
    "date_str": ["2024-01-15", "2024-02-20", "2024-03-10"],
    "value": [100, 200, 150]
})

raw["date"] = pd.to_datetime(raw["date_str"])
raw = raw.set_index("date").drop(columns="date_str")
print(type(raw.index))
raw

### 1c. Load CSV with `parse_dates` + `index_col`

This is the most common pattern when working with real data.  
We will load `airline_passengers.csv` and use it for the rest of this notebook.

In [ ]:
df = pd.read_csv(
    DATA_DIR / "airline_passengers.csv",
    parse_dates=["Month"],
    index_col="Month",
)
print(f"Index type : {type(df.index)}")
print(f"Shape      : {df.shape}")
print(f"Date range : {df.index.min()} to {df.index.max()}")
df.head()

In [ ]:
df.plot(title="Airline Passengers (1949-1960)", figsize=(10, 4))
plt.ylabel("Thousands of Passengers")
plt.tight_layout()
plt.show()

## 2. DatetimeIndex Properties

Once you have a `DatetimeIndex`, you can extract many useful components directly.

In [ ]:
idx = df.index

print("year       :", idx.year[:5].tolist())
print("month      :", idx.month[:5].tolist())
print("day        :", idx.day[:5].tolist())
print("dayofweek  :", idx.dayofweek[:5].tolist())   # Monday=0 ... Sunday=6
print("dayofyear  :", idx.dayofyear[:5].tolist())
print("quarter    :", idx.quarter[:5].tolist())

In [ ]:
# String name accessors
print("day_name()   :", idx.day_name()[:5].tolist())
print("month_name() :", idx.month_name()[:5].tolist())

In [ ]:
# Boolean boundary flags
print("is_month_start   :", idx.is_month_start[:5].tolist())
print("is_month_end     :", idx.is_month_end[:5].tolist())
print("is_quarter_start :", idx.is_quarter_start[:5].tolist())
print("is_year_end      :", idx.is_year_end[:5].tolist())

### Practical: extracting month from the index for seasonal analysis

In [ ]:
# Average passengers by month across all years
monthly_avg = df.groupby(df.index.month).mean()
monthly_avg.index = [
    "Jan", "Feb", "Mar", "Apr", "May", "Jun",
    "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"
]
monthly_avg.plot.bar(title="Average Passengers by Month", figsize=(8, 4), legend=False)
plt.ylabel("Thousands of Passengers")
plt.tight_layout()
plt.show()

## 3. Frequency and `asfreq()`

A `DatetimeIndex` can carry a **frequency** attribute (`.freq`) that tells pandas
the expected spacing between observations. This is critical for resampling,
forecasting, and detecting missing periods.

In [ ]:
# Our airline data was loaded without an explicit frequency
print("Current freq:", df.index.freq)

In [ ]:
# Detect the frequency from the data
inferred = pd.infer_freq(df.index)
print("Inferred freq:", inferred)

In [ ]:
# Apply the frequency with asfreq()
# If there are missing periods, they become NaN rows
df_freq = df.asfreq("MS")  # Month-Start since airline dates are 1st of month
print("Freq after asfreq:", df_freq.index.freq)
print("Any NaNs introduced:", df_freq.isna().any().any())
df_freq.head()

In [ ]:
# Demonstration: what happens when periods are missing
# Drop a few rows then call asfreq — the gaps become NaN
df_gap = df.drop(df.index[[5, 6, 7]])  # remove Jun/Jul/Aug 1949
df_gap_freq = df_gap.asfreq("MS")
print("Rows with NaN after asfreq:")
df_gap_freq[df_gap_freq.isna().any(axis=1)]

## 4. Date-Based Slicing (Partial String Indexing)

One of the most powerful features of a `DatetimeIndex` is **partial string indexing**:
you can slice with incomplete date strings, and pandas figures out the range.

In [ ]:
# Slice by year — returns all rows in 1960
df["1960"]

In [ ]:
# Slice by year-month — returns just June 1955
df["1955-06"]

In [ ]:
# Slice by range — Jan 1955 through Dec 1955 (inclusive on both ends)
df["1955-01":"1955-12"]

In [ ]:
# Using .loc with date strings
df.loc["1958-06":"1958-12"]

In [ ]:
# .truncate() — keep only data between two dates
df.truncate(before="1955-01-01", after="1955-12-31")

## 5. Boolean Indexing with Dates

You can use the index properties for boolean masks — handy for selecting
specific months, weekdays, or years.

In [ ]:
# Select summer months only (June, July, August)
summer = df[df.index.month.isin([6, 7, 8])]
print(f"Summer rows: {len(summer)}")
summer.head(6)

In [ ]:
# Select weekdays only (Monday=0 through Friday=4)
# (The airline data is monthly so all days are the 1st, but this
# pattern is critical for daily data.)
weekday_mask = df.index.dayofweek < 5
weekdays = df[weekday_mask]
print(f"Weekday rows: {len(weekdays)} out of {len(df)}")

In [ ]:
# Select specific years
fifties = df[df.index.year.isin([1955, 1956, 1957])]
fifties.plot(title="Passengers in 1955-1957", figsize=(8, 3))
plt.tight_layout()
plt.show()

## 6. Multi-Column Datetime Parsing

Sometimes a CSV stores year, month, and day in **separate columns** rather than
a single date string. `pd.to_datetime()` can combine them.

In [ ]:
# Create a sample DataFrame with separate date components
multi_col = pd.DataFrame({
    "year":  [2024, 2024, 2024, 2024],
    "month": [1, 2, 3, 4],
    "day":   [15, 20, 10, 5],
    "sales": [100, 200, 150, 300]
})
print("Before:")
print(multi_col)
print()

In [ ]:
# Combine year/month/day into a DatetimeIndex
multi_col.index = pd.to_datetime(multi_col[["year", "month", "day"]])
multi_col = multi_col.drop(columns=["year", "month", "day"])
print("After:")
print(multi_col)
print(f"\nIndex type: {type(multi_col.index)}")

## 7. DatetimeIndex vs RangeIndex

A very common mistake when loading time series data is forgetting `parse_dates`
and `index_col`. The result is a plain `RangeIndex` (0, 1, 2, ...) with dates
stored as strings — none of the slicing or resampling features will work.

In [ ]:
# WRONG: loading without parse_dates / index_col
df_bad = pd.read_csv(DATA_DIR / "airline_passengers.csv")
print(f"Index type : {type(df_bad.index)}")
print(f"Month dtype: {df_bad['Month'].dtype}")
df_bad.head(3)

In [ ]:
# RIGHT: loading with parse_dates + index_col
df_good = pd.read_csv(
    DATA_DIR / "airline_passengers.csv",
    parse_dates=["Month"],
    index_col="Month",
)
print(f"Index type: {type(df_good.index)}")
df_good.head(3)

In [ ]:
# Quick check you can use anywhere in your code
def check_datetime_index(dataframe: pd.DataFrame) -> None:
    """Assert that the DataFrame has a DatetimeIndex."""
    if not isinstance(dataframe.index, pd.DatetimeIndex):
        raise TypeError(
            f"Expected DatetimeIndex, got {type(dataframe.index).__name__}. "
            "Re-load with parse_dates and index_col."
        )
    print("Index is a valid DatetimeIndex.")

check_datetime_index(df_good)

In [ ]:
# This will raise a TypeError as expected
try:
    check_datetime_index(df_bad)
except TypeError as e:
    print(f"Caught: {e}")

---

## Key Takeaways

| Concept | Function / Pattern |
|---|---|
| Generate dates | `pd.date_range(start, periods, freq)` |
| Parse strings to datetime | `pd.to_datetime()` |
| Load CSV with datetime index | `pd.read_csv(..., parse_dates=[col], index_col=col)` |
| Extract components | `.year`, `.month`, `.quarter`, `.day_name()` |
| Set / detect frequency | `df.asfreq('ME')`, `pd.infer_freq(idx)` |
| Partial string slicing | `df['2024']`, `df['2024-06':'2024-12']` |
| Boolean date filtering | `df[df.index.month.isin([6, 7, 8])]` |
| Validate index type | `isinstance(df.index, pd.DatetimeIndex)` |